# aqui vamos trabalhar com a planilha

#"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\Base de dados\planilhas Geradas\planilha com as tags completa_12mil.csv"

# e filtrar com as tags 

Produção", "Bacia", "FPSO", "Operações","Plataforma","Pré-sal", "Exploração","FPSO", "E e P Unidade de Operações","Bacia terrestre","Perfuração","Empregados", "Estaleiro", "Exploração"}


In [1]:
import pandas as pd
import unicodedata

# Caminho do arquivo
arquivo = r"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\Base de dados\planilhas Geradas\planilha com as tags completa_12mil.csv"

# Função para normalizar texto (remove acento, baixa caixa, padroniza espaços)
def normalizar(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('utf-8')
    texto = texto.replace('_', ' ')
    return texto.strip()

# Lista de tags desejadas (com variações)
tags_desejadas = {"Produção","Operações","Plataforma","Pré-sal", "Exploração","FPSO", "E e P Unidade de Operações","Perfuração","Empregados", "Estaleiro", "Exploração"}

# Normalizar lista de referência
tags_normalizadas = {normalizar(tag) for tag in tags_desejadas}

# Ler CSV
df = pd.read_csv(
    arquivo,
    sep=';',
    encoding='utf-8',
    engine='python',
    on_bad_lines='skip'
)

# Função de filtro
def contem_tag(lista_tags):
    if pd.isna(lista_tags):
        return False
    tags = [normalizar(t) for t in str(lista_tags).split('|')]
    return any(tag in tags_normalizadas for tag in tags)

# Aplicar filtro
df_filtrado = df[df['tags'].apply(contem_tag)]

# Salvar resultado
saida = r"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\Base de dados\linhas_filtradas_por_tags.csv"
df_filtrado.to_csv(saida, index=False, encoding='utf-8-sig')

# Resultado
print(f"Linhas filtradas: {df_filtrado.shape[0]}")
print(f"Shape final: {df_filtrado.shape}")

Linhas filtradas: 7578
Shape final: (7578, 4)


# 🔧 ETAPA 0 — Instalação
📦 transformers

Biblioteca da Hugging Face

Contém o modelo BLIP que você está usando
Serve para:
carregar modelo (BlipForConditionalGeneration)
carregar processor/tokenizer (BlipProcessor)
É o coração do modelo de caption
📦 datasets

Também da Hugging Face

Facilita criar datasets estruturados
Permite:
transformar seus dados em formato padrão
integrar com o treinamento
Muito útil para fine-tuning
📦 pandas
Manipulação de dados tabulares (Excel)
Você usa para:
ler sua planilha
organizar colunas como descrição, links, etc.
📦 Pillow (PIL)
Trabalhar com imagens
Usado para:
abrir imagens baixadas
converter formatos
Essencial para alimentar o modelo
📦 requests
Fazer requisições HTTP
Você usa para:
baixar imagens da internet
⚠️ Aqui é um ponto sensível (erros 403, timeout, etc.)
📦 beautifulsoup4
Parser de HTML
Serve para:
extrair URLs de imagens das páginas
⚠️ Outro ponto crítico (depende da estrutura do site)

In [2]:
!pip install -q transformers datasets torch torchvision pillow requests tqdm
!{sys.executable} -m pip install datasets
!pip install -q  datasets

'{sys.executable}' nÆo ‚ reconhecido como um comando interno
ou externo, um programa oper vel ou um arquivo em lotes.


# ETAPA 1 — Carregar sua base (Excel com URLs)¶
 

In [3]:
# planilha 

import pandas as pd

caminho = r"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\Base de dados\planilhas Geradas\linhas_filtradas_por_tags.csv"

# como é CSV, use read_csv
df = pd.read_csv(caminho)

# selecionar colunas
df = df[['link', 'codigo', 'descricao', 'tags']]

# remover valores nulos
df = df.dropna().reset_index(drop=True)

print("Total registros:", len(df))
print(df.head())

Total registros: 7561
                                                link    codigo  \
0  https://bancodeimagens.petrobras.com.br/fotowe...  Dig47450   
1  https://bancodeimagens.petrobras.com.br/fotowe...  Dig47449   
2  https://bancodeimagens.petrobras.com.br/fotowe...  Dig47448   
3  https://bancodeimagens.petrobras.com.br/fotowe...  Dig47447   
4  https://bancodeimagens.petrobras.com.br/fotowe...  Dig47446   

                                  descricao  \
0  Plataforma PGP-1, conhecida como Garoupa   
1  Plataforma PGP-1, conhecida como Garoupa   
2  Plataforma PGP-1, conhecida como Garoupa   
3  Plataforma PGP-1, conhecida como Garoupa   
4  Plataforma PGP-1, conhecida como Garoupa   

                                                tags  
0  Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...  
1  Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...  
2  Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...  
3  Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...  
4  Bacia marítima|E&P|E

# 🌐 ETAPA 2 — Extrair URL real da imagem (CORREÇÃO CRÍTICA)
apenas a função extrair_url_imagem

In [4]:
#Solução
#data-src → melhor qualidade
#src → versão menor
#✅ Solução correta: pegar data-src



import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def extrair_url_imagem(link):
    try:
        r = requests.get(link, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        # 1. encontra o link .jpg.info
        for a in soup.find_all("a", href=True):
            href = a["href"]
            
            if ".jpg.info" in href:
                url_info = urljoin(link, href)

                # acessa a página .info
                r2 = requests.get(url_info, timeout=10)
                soup2 = BeautifulSoup(r2.text, "html.parser")

                # pega a imagem correta pelo ID
                img = soup2.find("img", {"id": "previewInitImage"})
                
                if img:
                    # PRIORIDADE: data-src (imagem maior)
                    if img.get("data-src"):
                        return urljoin(url_info, img["data-src"])
                    
                    # fallback: src
                    if img.get("src"):
                        return urljoin(url_info, img["src"])

        return None

    except:
        return None


# TESTE CRÍTICO
link = df["link"].iloc[0]

url = extrair_url_imagem(link)

print("URL extraída:", url)

if url:
    try:
        r = requests.get(url, timeout=10)
        
        print("Status:", r.status_code)
        print("Content-Type:", r.headers.get("content-type"))
        
        if "image" in r.headers.get("content-type", ""):
            print("✔ É uma imagem válida")
        else:
            print("⚠ Não parece ser uma imagem")
    
    except requests.exceptions.RequestException as e:
        print("Erro na requisição:", e)
else:
    print("⚠ URL não encontrada")

URL extraída: https://bancodeimagens.petrobras.com.br/fotoweb/cache/v2/u/f/Folder%203/Dig47450.jpg.lyf59o7_MS3t47VjQA0A.gXZHP3KatK.jpg
Status: 200
Content-Type: image/jpeg
✔ É uma imagem válida


Essa etapa faz duas coisas, de forma direta:

👉 1. Extração

Aplica a função extrair_url_imagem(link) para cada valor da coluna df["link"]
Resultado: uma nova coluna com URLs diretas das imagens

👉 2. Limpeza

Remove linhas onde:
a URL é None
ou não é uma imagem válida
Ou seja, elimina registros que não servem pro modelo
🔑 Em resumo

Transforma:

links de páginas → em links diretos de imagem
e mantém só os casos úteis para análise/modelo

# 🧹 ETAPA 3 — Extração e limpeza de URLs de imagem
Nesta etapa, aplicamos a função de extração a todos os links da base, obtendo URLs diretas de imagens e removendo registros inválidos.

👉 Esse código usa diretamente a função extrair_url_imagem(link) que você acabou de corrigir.

✔ O que ele faz, baseado no seu código anterior:
Para cada link:
chama extrair_url_imagem(link)
que agora:
entra na .info
pega o data-src
retorna a imagem em melhor resolução

In [ ]:
from tqdm import tqdm

# --- limitar base ---
df = df.head(7561).copy()

# --- lista ---
urls_validas = []

# --- loop com barra de progresso ---
for link in tqdm(df["link"], desc="Extraindo imagens"):

    try:
        url = extrair_url_imagem(link)
    except Exception:
        url = None

    urls_validas.append(url)

# --- adicionar coluna ---
df["url_valida"] = urls_validas

# --- limpeza ---
print("\nAntes da limpeza:", len(df))

df = df.dropna(subset=["url_valida"]).reset_index(drop=True)

print("Depois da limpeza:", len(df))
print("Removidos:", len(urls_validas) - len(df))

# --- amostra ---
print("\nAmostra:")
print(df[["link", "url_valida"]].head())

Extraindo imagens:  37%|████████████████████▎                                  | 2792/7561 [1:28:56<2:20:35,  1.77s/it]

🔑 Interpretação direta
7561 → total original
5955 → com imagem válida
1606 → removidos

# ETAPA 4 — Baixar imagens ✅

link → extrair_url_imagem → url_valida → imagem real

In [7]:
import os
import requests

# criar pasta para salvar imagens
pasta = "imagens_5955"
os.makedirs(pasta, exist_ok=True)

# lista para salvar caminhos locais
caminhos = []

total = len(df)

for i, url in enumerate(df["url_valida"]):
    print(f"Baixando {i+1}/{total}...")

    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            nome_arquivo = f"img_{i}.jpg"
            caminho = os.path.join(pasta, nome_arquivo)

            with open(caminho, "wb") as f:
                f.write(response.content)# salva URL original sem perda

            caminhos.append(caminho)
        else:
            caminhos.append(None)

    except:
        caminhos.append(None)

# adicionar ao dataframe
df["caminho_imagem"] = caminhos

print("\nDownload finalizado!")

Baixando 1/5955...
Baixando 2/5955...
Baixando 3/5955...
Baixando 4/5955...
Baixando 5/5955...
Baixando 6/5955...
Baixando 7/5955...
Baixando 8/5955...
Baixando 9/5955...
Baixando 10/5955...
Baixando 11/5955...
Baixando 12/5955...
Baixando 13/5955...
Baixando 14/5955...
Baixando 15/5955...
Baixando 16/5955...
Baixando 17/5955...
Baixando 18/5955...
Baixando 19/5955...
Baixando 20/5955...
Baixando 21/5955...
Baixando 22/5955...
Baixando 23/5955...
Baixando 24/5955...
Baixando 25/5955...
Baixando 26/5955...
Baixando 27/5955...
Baixando 28/5955...
Baixando 29/5955...
Baixando 30/5955...
Baixando 31/5955...
Baixando 32/5955...
Baixando 33/5955...
Baixando 34/5955...
Baixando 35/5955...
Baixando 36/5955...
Baixando 37/5955...
Baixando 38/5955...
Baixando 39/5955...
Baixando 40/5955...
Baixando 41/5955...
Baixando 42/5955...
Baixando 43/5955...
Baixando 44/5955...
Baixando 45/5955...
Baixando 46/5955...
Baixando 47/5955...
Baixando 48/5955...
Baixando 49/5955...
Baixando 50/5955...
Baixando 

Baixando 397/5955...
Baixando 398/5955...
Baixando 399/5955...
Baixando 400/5955...
Baixando 401/5955...
Baixando 402/5955...
Baixando 403/5955...
Baixando 404/5955...
Baixando 405/5955...
Baixando 406/5955...
Baixando 407/5955...
Baixando 408/5955...
Baixando 409/5955...
Baixando 410/5955...
Baixando 411/5955...
Baixando 412/5955...
Baixando 413/5955...
Baixando 414/5955...
Baixando 415/5955...
Baixando 416/5955...
Baixando 417/5955...
Baixando 418/5955...
Baixando 419/5955...
Baixando 420/5955...
Baixando 421/5955...
Baixando 422/5955...
Baixando 423/5955...
Baixando 424/5955...
Baixando 425/5955...
Baixando 426/5955...
Baixando 427/5955...
Baixando 428/5955...
Baixando 429/5955...
Baixando 430/5955...
Baixando 431/5955...
Baixando 432/5955...
Baixando 433/5955...
Baixando 434/5955...
Baixando 435/5955...
Baixando 436/5955...
Baixando 437/5955...
Baixando 438/5955...
Baixando 439/5955...
Baixando 440/5955...
Baixando 441/5955...
Baixando 442/5955...
Baixando 443/5955...
Baixando 444/

Baixando 788/5955...
Baixando 789/5955...
Baixando 790/5955...
Baixando 791/5955...
Baixando 792/5955...
Baixando 793/5955...
Baixando 794/5955...
Baixando 795/5955...
Baixando 796/5955...
Baixando 797/5955...
Baixando 798/5955...
Baixando 799/5955...
Baixando 800/5955...
Baixando 801/5955...
Baixando 802/5955...
Baixando 803/5955...
Baixando 804/5955...
Baixando 805/5955...
Baixando 806/5955...
Baixando 807/5955...
Baixando 808/5955...
Baixando 809/5955...
Baixando 810/5955...
Baixando 811/5955...
Baixando 812/5955...
Baixando 813/5955...
Baixando 814/5955...
Baixando 815/5955...
Baixando 816/5955...
Baixando 817/5955...
Baixando 818/5955...
Baixando 819/5955...
Baixando 820/5955...
Baixando 821/5955...
Baixando 822/5955...
Baixando 823/5955...
Baixando 824/5955...
Baixando 825/5955...
Baixando 826/5955...
Baixando 827/5955...
Baixando 828/5955...
Baixando 829/5955...
Baixando 830/5955...
Baixando 831/5955...
Baixando 832/5955...
Baixando 833/5955...
Baixando 834/5955...
Baixando 835/

Baixando 1171/5955...
Baixando 1172/5955...
Baixando 1173/5955...
Baixando 1174/5955...
Baixando 1175/5955...
Baixando 1176/5955...
Baixando 1177/5955...
Baixando 1178/5955...
Baixando 1179/5955...
Baixando 1180/5955...
Baixando 1181/5955...
Baixando 1182/5955...
Baixando 1183/5955...
Baixando 1184/5955...
Baixando 1185/5955...
Baixando 1186/5955...
Baixando 1187/5955...
Baixando 1188/5955...
Baixando 1189/5955...
Baixando 1190/5955...
Baixando 1191/5955...
Baixando 1192/5955...
Baixando 1193/5955...
Baixando 1194/5955...
Baixando 1195/5955...
Baixando 1196/5955...
Baixando 1197/5955...
Baixando 1198/5955...
Baixando 1199/5955...
Baixando 1200/5955...
Baixando 1201/5955...
Baixando 1202/5955...
Baixando 1203/5955...
Baixando 1204/5955...
Baixando 1205/5955...
Baixando 1206/5955...
Baixando 1207/5955...
Baixando 1208/5955...
Baixando 1209/5955...
Baixando 1210/5955...
Baixando 1211/5955...
Baixando 1212/5955...
Baixando 1213/5955...
Baixando 1214/5955...
Baixando 1215/5955...
Baixando 1

Baixando 1544/5955...
Baixando 1545/5955...
Baixando 1546/5955...
Baixando 1547/5955...
Baixando 1548/5955...
Baixando 1549/5955...
Baixando 1550/5955...
Baixando 1551/5955...
Baixando 1552/5955...
Baixando 1553/5955...
Baixando 1554/5955...
Baixando 1555/5955...
Baixando 1556/5955...
Baixando 1557/5955...
Baixando 1558/5955...
Baixando 1559/5955...
Baixando 1560/5955...
Baixando 1561/5955...
Baixando 1562/5955...
Baixando 1563/5955...
Baixando 1564/5955...
Baixando 1565/5955...
Baixando 1566/5955...
Baixando 1567/5955...
Baixando 1568/5955...
Baixando 1569/5955...
Baixando 1570/5955...
Baixando 1571/5955...
Baixando 1572/5955...
Baixando 1573/5955...
Baixando 1574/5955...
Baixando 1575/5955...
Baixando 1576/5955...
Baixando 1577/5955...
Baixando 1578/5955...
Baixando 1579/5955...
Baixando 1580/5955...
Baixando 1581/5955...
Baixando 1582/5955...
Baixando 1583/5955...
Baixando 1584/5955...
Baixando 1585/5955...
Baixando 1586/5955...
Baixando 1587/5955...
Baixando 1588/5955...
Baixando 1

...

Baixando 4155/5955...
Baixando 4156/5955...
Baixando 4157/5955...
Baixando 4158/5955...
Baixando 4159/5955...
Baixando 4160/5955...
Baixando 4161/5955...
Baixando 4162/5955...
Baixando 4163/5955...
Baixando 4164/5955...
Baixando 4165/5955...
Baixando 4166/5955...
Baixando 4167/5955...
Baixando 4168/5955...
Baixando 4169/5955...
Baixando 4170/5955...
Baixando 4171/5955...
Baixando 4172/5955...
Baixando 4173/5955...
Baixando 4174/5955...
Baixando 4175/5955...
Baixando 4176/5955...
Baixando 4177/5955...
Baixando 4178/5955...
Baixando 4179/5955...
Baixando 4180/5955...
Baixando 4181/5955...
Baixando 4182/5955...
Baixando 4183/5955...
Baixando 4184/5955...
Baixando 4185/5955...
Baixando 4186/5955...
Baixando 4187/5955...
Baixando 4188/5955...
Baixando 4189/5955...
Baixando 4190/5955...
Baixando 4191/5955...
Baixando 4192/5955...
Baixando 4193/5955...
Baixando 4194/5955...
Baixando 4195/5955...
Baixando 4196/5955...
Baixando 4197/5955...
Baixando 4198/5955...
Baixando 4199/5955...
Baixando 4

Baixando 4528/5955...
Baixando 4529/5955...
Baixando 4530/5955...
Baixando 4531/5955...
Baixando 4532/5955...
Baixando 4533/5955...
Baixando 4534/5955...
Baixando 4535/5955...
Baixando 4536/5955...
Baixando 4537/5955...
Baixando 4538/5955...
Baixando 4539/5955...
Baixando 4540/5955...
Baixando 4541/5955...
Baixando 4542/5955...
Baixando 4543/5955...
Baixando 4544/5955...
Baixando 4545/5955...
Baixando 4546/5955...
Baixando 4547/5955...
Baixando 4548/5955...
Baixando 4549/5955...
Baixando 4550/5955...
Baixando 4551/5955...
Baixando 4552/5955...
Baixando 4553/5955...
Baixando 4554/5955...
Baixando 4555/5955...
Baixando 4556/5955...
Baixando 4557/5955...
Baixando 4558/5955...
Baixando 4559/5955...
Baixando 4560/5955...
Baixando 4561/5955...
Baixando 4562/5955...
Baixando 4563/5955...
Baixando 4564/5955...
Baixando 4565/5955...
Baixando 4566/5955...
Baixando 4567/5955...
Baixando 4568/5955...
Baixando 4569/5955...
Baixando 4570/5955...
Baixando 4571/5955...
Baixando 4572/5955...
Baixando 4

Baixando 4901/5955...
Baixando 4902/5955...
Baixando 4903/5955...
Baixando 4904/5955...
Baixando 4905/5955...
Baixando 4906/5955...
Baixando 4907/5955...
Baixando 4908/5955...
Baixando 4909/5955...
Baixando 4910/5955...
Baixando 4911/5955...
Baixando 4912/5955...
Baixando 4913/5955...
Baixando 4914/5955...
Baixando 4915/5955...
Baixando 4916/5955...
Baixando 4917/5955...
Baixando 4918/5955...
Baixando 4919/5955...
Baixando 4920/5955...
Baixando 4921/5955...
Baixando 4922/5955...
Baixando 4923/5955...
Baixando 4924/5955...
Baixando 4925/5955...
Baixando 4926/5955...
Baixando 4927/5955...
Baixando 4928/5955...
Baixando 4929/5955...
Baixando 4930/5955...
Baixando 4931/5955...
Baixando 4932/5955...
Baixando 4933/5955...
Baixando 4934/5955...
Baixando 4935/5955...
Baixando 4936/5955...
Baixando 4937/5955...
Baixando 4938/5955...
Baixando 4939/5955...
Baixando 4940/5955...
Baixando 4941/5955...
Baixando 4942/5955...
Baixando 4943/5955...
Baixando 4944/5955...
Baixando 4945/5955...
Baixando 4

Baixando 5274/5955...
Baixando 5275/5955...
Baixando 5276/5955...
Baixando 5277/5955...
Baixando 5278/5955...
Baixando 5279/5955...
Baixando 5280/5955...
Baixando 5281/5955...
Baixando 5282/5955...
Baixando 5283/5955...
Baixando 5284/5955...
Baixando 5285/5955...
Baixando 5286/5955...
Baixando 5287/5955...
Baixando 5288/5955...
Baixando 5289/5955...
Baixando 5290/5955...
Baixando 5291/5955...
Baixando 5292/5955...
Baixando 5293/5955...
Baixando 5294/5955...
Baixando 5295/5955...
Baixando 5296/5955...
Baixando 5297/5955...
Baixando 5298/5955...
Baixando 5299/5955...
Baixando 5300/5955...
Baixando 5301/5955...
Baixando 5302/5955...
Baixando 5303/5955...
Baixando 5304/5955...
Baixando 5305/5955...
Baixando 5306/5955...
Baixando 5307/5955...
Baixando 5308/5955...
Baixando 5309/5955...
Baixando 5310/5955...
Baixando 5311/5955...
Baixando 5312/5955...
Baixando 5313/5955...
Baixando 5314/5955...
Baixando 5315/5955...
Baixando 5316/5955...
Baixando 5317/5955...
Baixando 5318/5955...
Baixando 5

Baixando 5647/5955...
Baixando 5648/5955...
Baixando 5649/5955...
Baixando 5650/5955...
Baixando 5651/5955...
Baixando 5652/5955...
Baixando 5653/5955...
Baixando 5654/5955...
Baixando 5655/5955...
Baixando 5656/5955...
Baixando 5657/5955...
Baixando 5658/5955...
Baixando 5659/5955...
Baixando 5660/5955...
Baixando 5661/5955...
Baixando 5662/5955...
Baixando 5663/5955...
Baixando 5664/5955...
Baixando 5665/5955...
Baixando 5666/5955...
Baixando 5667/5955...
Baixando 5668/5955...
Baixando 5669/5955...
Baixando 5670/5955...
Baixando 5671/5955...
Baixando 5672/5955...
Baixando 5673/5955...
Baixando 5674/5955...
Baixando 5675/5955...
Baixando 5676/5955...
Baixando 5677/5955...
Baixando 5678/5955...
Baixando 5679/5955...
Baixando 5680/5955...
Baixando 5681/5955...
Baixando 5682/5955...
Baixando 5683/5955...
Baixando 5684/5955...
Baixando 5685/5955...
Baixando 5686/5955...
Baixando 5687/5955...
Baixando 5688/5955...
Baixando 5689/5955...
Baixando 5690/5955...
Baixando 5691/5955...
Baixando 5

imagem local + texto → pronto para treino

👉 Isso é exatamente o que modelos da Hugging Face esperam



In [8]:
df.head(5)

,link,codigo,descricao,tags,url_valida,caminho_imagem
0,https://bancodeimagens.petrobras.com.br/fotowe...,Dig47450,"Plataforma PGP-1, conhecida como Garoupa",Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...,https://bancodeimagens.petrobras.com.br/fotowe...,imagens_5955\img_0.jpg
1,https://bancodeimagens.petrobras.com.br/fotowe...,Dig47449,"Plataforma PGP-1, conhecida como Garoupa",Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...,https://bancodeimagens.petrobras.com.br/fotowe...,imagens_5955\img_1.jpg
2,https://bancodeimagens.petrobras.com.br/fotowe...,Dig47448,"Plataforma PGP-1, conhecida como Garoupa",Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...,https://bancodeimagens.petrobras.com.br/fotowe...,imagens_5955\img_2.jpg
3,https://bancodeimagens.petrobras.com.br/fotowe...,Dig47447,"Plataforma PGP-1, conhecida como Garoupa",Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...,https://bancodeimagens.petrobras.com.br/fotowe...,imagens_5955\img_3.jpg
4,https://bancodeimagens.petrobras.com.br/fotowe...,Dig47446,"Plataforma PGP-1, conhecida como Garoupa",Bacia marítima|E&P|E&P Produção|Garoupa|Offsho...,https://bancodeimagens.petrobras.com.br/fotowe...,imagens_5955\img_4.jpg


# 🧱 ETAPA 5 — Montar dataset (imagem + texto) → cria os dados

In [9]:
print(df.columns.tolist())
print(df.shape)

['link', 'codigo', 'descricao', 'tags', 'url_valida', 'caminho_imagem']
(5955, 6)


In [22]:
df_dataset = df[["caminho_imagem", "descricao"]].copy()

df_dataset.columns = ["image_path", "text"]

df_dataset = df_dataset.dropna().reset_index(drop=True)

print("Tamanho do dataset:", len(df_dataset))
df_dataset.head()

Tamanho do dataset: 5726


,image_path,text
0,imagens_5955\img_0.jpg,"Plataforma PGP-1, conhecida como Garoupa"
1,imagens_5955\img_1.jpg,"Plataforma PGP-1, conhecida como Garoupa"
2,imagens_5955\img_2.jpg,"Plataforma PGP-1, conhecida como Garoupa"
3,imagens_5955\img_3.jpg,"Plataforma PGP-1, conhecida como Garoupa"
4,imagens_5955\img_4.jpg,"Plataforma PGP-1, conhecida como Garoupa"


 essa diferença é normal e esperada.

🔑 O que aconteceu

Antes você tinha:

5955 linhas (após extração)

Agora:

5726 linhas

👉 Diferença: 229 linhas

✔ Motivo direto

Aqui:

df_dataset = df_dataset.dropna()

👉 Isso remove linhas onde:

caminho_imagem é None (falha no download)
ou descricao está vazia ⚠

# 🤖 ETAPA 6 — Preparar dados para o modelo (BLIP)
Aqui você vai:

carregar imagens processar texto preparar inputs para o modelo da Hugging Face

In [13]:
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

print("Modelo carregado com sucesso!")

C:\Users\berna\GitHub\XP DESAFIO FINAL outubro de 2025\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████████████████████████████████████| 473/473 [00:00<00:00, 14597.31it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` t

Modelo carregado com sucesso!


# 🚀 ETAPA 7 — Preparação para treino

🧠 O que você quer fazer (confirmando)

Você disse:

“quero que o modelo veja a imagem e gere uma descrição sozinho”

👉 Isso significa:

Entrada: imagem
Saída: texto
⚠️ Onde você está agora

Seu dataset tem:

imagem ✔️
descrição ✔️
tags ✔️

In [16]:
#Testar uma imagem no modelo
from PIL import Image

# pegar 1 exemplo
img_path = df_dataset["image_path"].iloc[0]
texto = df_dataset["text"].iloc[0]

image = Image.open(img_path).convert("RGB")

# processar
inputs = processor(images=image, text=texto, return_tensors="pt")

print("Inputs gerados com sucesso!")

Inputs gerados com sucesso!


In [16]:
from torch.utils.data import Dataset

class MeuDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.df = dataframe
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        from PIL import Image
        
        img_path = self.df.iloc[idx]["image_path"]
        texto = self.df.iloc[idx]["text"]

        image = Image.open(img_path).convert("RGB")

        inputs = self.processor(
            images=image,
            text=texto,
            return_tensors="pt",
            padding="max_length",
            truncation=True
        )

        # remove dimensão extra
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        return inputs

In [17]:
#  Preparação para treino

from PIL import Image

# pegar 1 exemplo do dataset
img_path = df_dataset["image_path"].iloc[0]
texto = df_dataset["text"].iloc[0]

try:
    # 1. abrir a imagem
    image = Image.open(img_path).convert("RGB")

    # 2. transformar imagem + texto em números
    inputs = processor(
        images=image,
        text=texto,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # 3. ver o resultado
    print("Inputs gerados com sucesso!")
    print("Chaves:", inputs.keys())
    
    # ver formatos (muito importante)
    for k, v in inputs.items():
        print(f"{k}: shape {v.shape}")

except FileNotFoundError:
    print(f"Erro: imagem não encontrada em {img_path}")

except Exception as e:
    print(f"Erro ao processar: {e}")

Inputs gerados com sucesso!
Chaves: KeysView({'pixel_values': tensor([[[[-1.4127, -1.4565, -1.4711,  ..., -1.3543, -1.3835, -1.4127],
          [-1.4419, -1.4565, -1.4419,  ..., -1.3397, -1.3689, -1.3981],
          [-1.4711, -1.4419, -1.4711,  ..., -1.3105, -1.3397, -1.3689],
          ...,
          [ 0.9668, -0.0988, -0.4784,  ...,  1.8865,  1.9157,  1.9303],
          [ 0.8063, -0.2594, -0.5076,  ...,  1.9011,  1.9157,  1.9303],
          [ 0.6895, -0.3470, -0.4200,  ...,  1.9157,  1.9303,  1.9157]],

         [[-0.8666, -0.8967, -0.8967,  ..., -1.3619, -1.3919, -1.4219],
          [-0.8816, -0.9117, -0.8816,  ..., -1.3769, -1.4069, -1.4369],
          [-0.9417, -0.9567, -1.0167,  ..., -1.3919, -1.4219, -1.4519],
          ...,
          [ 0.1839, -0.7166, -1.1368,  ...,  1.5946,  1.5796,  1.5646],
          [ 0.0338, -0.8516, -1.1368,  ...,  1.6247,  1.6247,  1.5946],
          [-0.0562, -0.9117, -1.0467,  ...,  1.6547,  1.6547,  1.6247]],

         [[-0.1435, -0.1720, -0.1720,  .

# 🚀 ETAPA 8 — Treino simples (primeira execução)  

In [18]:
df_train = df_dataset.copy()

✔ 6. Conclusão direta
✔ Não precisa rodar de novo completo
✔ Não precisa 3 épocas
✔ Seu modelo já aprendeu bem
✔ O problema é só performance

👉 Direto ao ponto (qualidade do treino):

🔑 4 vs 8 — Qualidade
✔ batch = 4
✔ melhor generalização
✔ aprende mais “detalhes”
✔ mais robusto
✔ batch = 8
✔ mais estável
✔ converge mais suave
❌ pode generalizar um pouco pior
🧠 Intuição simples
batch 4 → modelo “erra mais” → aprende mais nuances
batch 8 → modelo “suaviza” → aprende mais estável

Resumo final
Imagens: 1000
Batch: 4
Épocas: 1
Steps: ~250

In [24]:
#O mesmo código, so que com 100 linhas escolhidas de forma radomica
 # batch_size=4,

# 🔥 AMOSTRAGEM ALEATÓRIA (1000 imagens dos 5700)
df_treino = df.sample(1000, random_state=42).copy()

# 🔥 preparar dataset com descricao + tags
df_dataset = df_treino[["caminho_imagem", "descricao", "tags"]].copy()

df_dataset = df_dataset.rename(columns={
    "caminho_imagem": "image_path"
})

df_dataset = df_dataset.dropna().reset_index(drop=True)

print("Tamanho do dataset:", len(df_dataset))


# =========================
# TREINO
# =========================
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image

# =========================
# DATASET
# =========================
class MeuDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        # 🔥 texto mais rico
        text = row["descricao"] + ". Tags: " + str(row["tags"])

        inputs = processor(
            images=image,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=100
        )

        inputs = {k: v.squeeze(0) for k, v in inputs.items()}

        return inputs


# =========================
# DATALOADER
# =========================
dataset = MeuDataset(df_dataset)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True
)

# =========================
# TREINO
# =========================
model.train()

optimizer = AdamW(model.parameters(), lr=5e-5)

for epoch in range(1):
    print(f"\nÉpoca {epoch+1}")

    for step, batch in enumerate(dataloader):
        try:
            outputs = model(**batch, labels=batch["input_ids"])
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            if step % 20 == 0:
                print(f"Step {step} - Loss: {loss.item()}")

        except Exception as e:
            print(f"Erro no batch {step}: {e}")

Tamanho do dataset: 962

Época 1
Step 0 - Loss: 1.3057801723480225
Step 20 - Loss: 1.1291155815124512
Step 40 - Loss: 0.9527479410171509
Step 60 - Loss: 0.7456846833229065
Step 80 - Loss: 0.6567469835281372
Step 100 - Loss: 0.2474520206451416
Step 120 - Loss: 0.3093552589416504
Step 140 - Loss: 0.3695811331272125
Step 160 - Loss: 0.4070103168487549
Step 180 - Loss: 0.35089343786239624
Step 200 - Loss: 0.34053876996040344
Step 220 - Loss: 0.43511253595352173
Step 240 - Loss: 0.3171859085559845


🔑 O que aconteceu no seu treino

👉 O modelo SIM viu a imagem ✔
👉 E também viu o texto (descrição + tags) ✔

🧠 Como ele aprendeu

Em cada passo, você deu:

Imagem  →  "Plataforma PGP-1... Tags: petróleo, offshore..."

👉 então ele aprende:

👉 associar imagem → texto

⚠️ Mas tem um detalhe MUITO importante

👉 você deu o texto já pronto

Então o modelo aprende:

👉 “quando vejo essa imagem → esse tipo de descrição”

🎯 O objetivo do treino

👉 ensinar o modelo a fazer:

Imagem → gerar descrição sozinho
################################################################################

Por isso o teste funciona

Quando você faz:

inputs = processor(images=image, return_tensors="pt")

👉 você NÃO passa texto

👉 aí o modelo tenta gerar sozinho 
###################################################################################

🔑 O que esses números mostram
✔ Começo
Loss: 1.30 → 0.95 → 0.74

👉 modelo aprendendo rápido ✅

✔ Meio
Loss: 0.65 → 0.24

👉 aprendizado forte (excelente) 🚀

✔ Final
Loss: 0.30 ~ 0.43

👉 estabilizou (normalíssimo)

🎯 Interpretação correta

👉 Isso aqui é o ideal:

caiu rápido ✔
estabilizou ✔
não explodiu ✔

👉 treino saudável

⚠️ Ponto importante

👉 Loss 0.24 foi o melhor momento
Depois subiu um pouco:

👉 isso pode indicar:

leve overfitting
ou variação normal de batch

👉 nada preocupante

# validação + teste

✔ Correto agora

👉 validar e testar em outro conjunto aleatório (diferente)

⚠️ Importante

👉 NÃO pode usar as mesmas 1000 do treino
senão vira "cola" (overfitting)

Perfeito 👍 — agora vamos fazer validação e teste exatamente em cima do seu treino, sem mudar nada de lógica.

👉 Mantendo:

PIL.Image.open() ✔
processor() ✔
mesmo formato


Perfeito 👍 — vamos manter EXATAMENTE o mesmo padrão que você usou no treino (PIL + processor).

Segue o bloco completo de VALIDAÇÃO + TESTE, usando seu estilo 👇

# ETAPA 09  validação + teste  

# 1000 AMOSTRAS ALEATÓRIAS (fora do treino) /  SPLIT (500 validação / 500 teste)

In [27]:
 #1000 AMOSTRAS ALEATÓRIAS (fora do treino) /  SPLIT (500 validação / 500 teste)

# =========================
# 1000 AMOSTRAS ALEATÓRIAS (fora do treino)
# =========================
df_restante = df.drop(df_treino.index)

df_sample = df_restante.sample(1000, random_state=123).copy()

# montar texto igual ao treino
df_sample["text"] = df_sample["descricao"] + ". Tags: " + df_sample["tags"].astype(str)

df_sample = df_sample[["caminho_imagem", "text"]].dropna().reset_index(drop=True)

print("Dataset amostra:", len(df_sample))


# =========================
# SPLIT (500 validação / 500 teste)
# =========================
df_val = df_sample[:500]
df_test = df_sample[500:]

print("Validação:", len(df_val))
print("Teste:", len(df_test))


# =========================
# VALIDAÇÃO (LOSS)
# =========================
from PIL import Image
import torch

model.eval()

val_loss_total = 0

for i in range(len(df_val)):
    row = df_val.iloc[i]

    try:
        image = Image.open(row["caminho_imagem"]).convert("RGB")
        text = row["text"]

        inputs = processor(
            images=image,
            text=text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=100
        )

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            val_loss_total += outputs.loss.item()

        if i % 50 == 0:
            print(f"Validação: {i}/{len(df_val)}")

    except Exception as e:
        print(f"Erro validação {i}: {e}")

val_loss = val_loss_total / len(df_val)

print("\n✔ Val Loss:", val_loss)


# =========================
# TESTE (GERAR TEXTO)
# =========================
print("\n===== TESTE =====")

for i in range(5):
    row = df_test.iloc[i]

    try:
        image = Image.open(row["caminho_imagem"]).convert("RGB")

        inputs = processor(images=image, return_tensors="pt")

        with torch.no_grad():
            output = model.generate(**inputs, max_length=100)

        caption = processor.decode(output[0], skip_special_tokens=True)

        print("\n--- EXEMPLO ---")
        print("Real:", row["text"])
        print("Gerado:", caption)

    except Exception as e:
        print(f"Erro teste {i}: {e}")

Dataset amostra: 961
Validação: 500
Teste: 461
Validação: 0/500
Validação: 50/500
Validação: 100/500
Validação: 150/500
Validação: 200/500
Validação: 250/500
Validação: 300/500
Validação: 350/500
Validação: 400/500
Validação: 450/500

✔ Val Loss: 0.38504129894077777

===== TESTE =====

--- EXEMPLO ---
Real: Conversão do navio de produção FPSO P-50  no estaleiro Mauá-Jurong. Tags: Estaleiros|Construção|Engenharia|P.50|FPSO|Produção|Indústria_naval
Gerado: conversao do navio de producao ( fpso ) p. 50 no estaleiro maua - jurong. tags : construcao | engenharia | p. 50 | fpso | diversos | estaleiro | construcao _ naval | e _ e _ p _ producao

--- EXEMPLO ---
Real: Homens trabalhando no navio de produção (FPSO) Plataforma P-54. Tags: FPSO|P.54|Empregados_produção|Empregados|Plataforma_de_Produção|E_e_P_Produção
Gerado: o presidente da republica luiz inacio lula da silva e a cerimonia de inicio da producao do navio de producao fpso p - 50, no estaleiro maua - jurong. tags : construcao | p. 5

📊 🔍 1. Seu Val Loss

👉 0.38

✔ Isso é BOM
✔ Modelo aprendeu o padrão
✔ Não está explodindo nem travado

👉 Regra prática:

~10 → ruim (início)
~1 → ok
< 0.5 → bom
< 0.3 → muito bom

👉 Você está no caminho certo

🧠 🔎 2. O que o modelo aprendeu
✔ Ele entendeu:
Estrutura de frase
Linguagem técnica (FPSO, produção, etc.)
Formato "descrição + tags"

👉 EXEMPLO:

"conversao do navio de producao fpso..."

✔ Muito próximo do real

⚠️ 🔥 3. Problema principal (IMPORTANTE)

O modelo está:

❌ Misturando exemplos

Ex:

P-54 → virou P-50
local errado
descrição de outra imagem

👉 Isso é clássico:

🧠 o modelo "decorou padrões", não a imagem em si totalmente

🎯 💡 Por que isso acontece?

3 motivos principais:

1. Dataset muito parecido

Muitas imagens de:

plataformas
FPSO
trabalhadores

👉 modelo confunde

2. Texto muito parecido

Ex:

"Plataforma P-50..."
"Plataforma P-54..."

👉 diferença pequena → modelo troca

3. Treino curto (1 época)

👉 ele aprendeu padrão geral, mas não refinou detalhes

# 🏋️ — Treinamento (fine-tuning) o que é ?

🧠 O que é Fine-tuning (Treinamento)?

👉 Fine-tuning é:

pegar um modelo já pronto e adaptar ele para o seu problema

📌 No seu caso

Você usou o modelo:

👉 Salesforce/blip-image-captioning-base

Esse modelo já sabia:

reconhecer objetos
gerar descrições genéricas de imagens
🔥 O que você fez na prática

Você ensinou o modelo:

“Descreva imagens no contexto da indústria de petróleo”

👉 com seus dados:

imagem ✔
descrição ✔
tags ✔
🎯 Tradução simples
Antes do treino:

Imagem →
👉 "a large structure in the ocean"

Depois do seu fine-tuning:

Imagem →
👉 "Plataforma FPSO utilizada na produção de petróleo em alto mar..."

🧠 Analogia simples

Pensa assim:

Modelo base = pessoa que sabe inglês
Fine-tuning = ensinar vocabulário técnico

👉 você transformou ele em:

"especialista em petróleo e plataformas"

⚙️ Tecnicamente (sem complicar)

Durante o treino:

Modelo vê imagem + texto
Ele tenta prever o texto
Erra → calcula loss
Ajusta os pesos (aprende)

👉 isso acontece milhares de vezes (steps)

🔢 Onde entram época e step
Step = 1 atualização do modelo
Época = passar por TODO dataset

👉 você fez:

~240 steps (1000 imagens / batch 4)
1 época → viu tudo 1 vez
⚠️ Importante

Fine-tuning não cria um modelo do zero

👉 ele:

aproveita conhecimento existente
adapta pro seu domínio

# 💾 ✅ETAPA 10 SALVAR MODELO + PROCESSOR

In [28]:
# =========================
# SALVAR MODELO
# =========================
pasta_modelo = "meu_modelo_blip_31_03_2026"

model.save_pretrained(pasta_modelo)
processor.save_pretrained(pasta_modelo)

print("✔ Modelo salvo com sucesso!")

Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:05<00:00,  5.05s/it]


✔ Modelo salvo com sucesso!


In [29]:
import os
print(os.getcwd())

C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\notebooks rascunho
